# Belle II Continuum Suppression

This notebook is the project's sequential learning artifact. It explains the physics and orchestrates tested functions from `src/` in this order: data ingestion → audit → cleaning → EDA → splitting → baselines → six classifiers → feature reduction → evaluation.

The raw dataset is private and is never committed. Until it is placed in `data/raw/`, data-dependent cells explain what is waiting instead of failing.

## 1. Imports and reproducibility

Reusable behavior belongs in `src/`; this notebook imports and demonstrates it. A single random seed and test fraction will be reused by every model for a fair comparison.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    DB_PATH, RANDOM_STATE, RAW_DATA_DIR, TEST_SIZE, VALIDATION_SIZE
)
from src.data_io import assign_split, audit_frame, load_raw_to_db
from src.db import read_table, table_exists

print(f"Project root: {PROJECT_ROOT}")
print(
    f"Random state: {RANDOM_STATE}; split fractions: "
    f"train={1 - VALIDATION_SIZE - TEST_SIZE:.0%}, "
    f"validation={VALIDATION_SIZE:.0%}, test={TEST_SIZE:.0%}"
)

## 2. Data provenance

The instructor did not supply an analysis-level CSV. We therefore use the checksum-verified official Belle II MDST files `Bd2K0spi0_to_train.root` (signal) and `ccbar_sample_to_train.root` (charm-continuum background). MDST files are detector-analysis inputs, not ready-made pandas tables: `scripts/generate_belle2_ntuple.py` must reconstruct the B candidate, rest of event, and continuum-suppression observables with `basf2`. Each class is processed separately so its label and source provenance are explicit.

```bash
basf2 scripts/generate_belle2_ntuple.py -- \
  --input data/raw/Bd2K0spi0_to_train.root \
  --output data/interim/signal_ntuple.root --label 1
basf2 scripts/generate_belle2_ntuple.py -- \
  --input data/raw/ccbar_sample_to_train.root \
  --output data/interim/background_ntuple.root --label 0
```

First validate both commands with `--max-events 100`; omit that option only for the final human-run processing. Candidate construction follows Belle II's B2A711 example for these exact samples: standard K-short and pi0 lists, tight B-candidate kinematics, MC matching, and `isSignal or isContinuumEvent`. Thus simulation truth removes incorrectly reconstructed signal candidates but is not claimed as a selection available on collision data. The script fixes the basf2 random seed and prints processing statistics for a reproducible, auditable extraction. The commands are intentionally not executed by this notebook because full MDST reconstruction is long-running. `VariablesToNtuple` automatically records the canonical `__experiment__`, `__run__`, `__event__`, `__production__`, and `__candidate__` identifier branches. The private ROOT outputs will be audited and combined into one canonical labeled CSV in the next step. In current Belle II releases, `KSFWVariables(pt_sum)` is the feature called `KSFWVariables(et)` in older material. The present background covers only charm continuum, so conclusions cannot yet be assumed to represent all light-quark continuum flavors.

## 3. Locate the private dataset

The untouched CSV belongs in `data/raw/`. Exactly one CSV is selected automatically; if none or several are present, the notebook stops the data-dependent sequence safely and explains what to fix.

In [ ]:
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))
raw_csv_path = csv_files[0] if len(csv_files) == 1 else None

if not csv_files:
    print("Waiting for the private Belle II CSV in data/raw/.")
elif len(csv_files) > 1:
    print("Multiple CSV files found; select the intended source explicitly before ingestion:")
    for path in csv_files:
        print(f"  - {path.name}")
else:
    print(f"Dataset found: {raw_csv_path.name}")

## 4. Immutable CSV → SQLite ingestion

`raw_events` is the database copy of the source and must not be silently replaced. On later runs, we read the existing table instead of rewriting it. This demonstrates database writing and reading while preserving raw-data provenance.

In [ ]:
raw_events = None
if raw_csv_path is not None:
    if table_exists("raw_events", DB_PATH):
        raw_events = read_table("raw_events", DB_PATH)
        print("Read the existing immutable raw_events table.")
    else:
        raw_events = load_raw_to_db(raw_csv_path, DB_PATH)
        print("Wrote the immutable raw_events table.")
    print(f"Loaded {len(raw_events):,} rows and {raw_events.shape[1]} columns.")
else:
    print("Ingestion skipped until exactly one source CSV is available.")

## 5. Raw-data audit before cleaning

We inspect shape, types, missing and non-finite values, duplicates, and constant columns without changing any event. These facts will drive explicit cleaning decisions after the real schema is known.

In [ ]:
raw_audit = audit_frame(raw_events) if raw_events is not None else None
if raw_audit is not None:
    print(f"Rows: {raw_audit['rows']:,}; columns: {raw_audit['columns']}")
    print(f"Duplicate rows: {raw_audit['duplicate_rows']:,}")
    print(f"Constant columns: {raw_audit['constant_columns']}")
    display(raw_events.head())
else:
    print("Audit skipped because no dataset has been loaded.")

## 6. Persist one fair train/validation/test split

The label name and encoding must be confirmed from the actual dataset—never guessed. After confirmation, `assign_split` creates one stratified 70/15/15 assignment that every classifier reuses. Training rows fit models and learned preprocessing; validation rows guide hyperparameters, thresholds, feature sets, and model selection; test rows remain untouched until final evaluation.

In [ ]:
LABEL_COLUMN = None  # Set only after the real schema and physics meaning are confirmed.
events_with_split = None
if raw_events is not None and LABEL_COLUMN is not None:
    events_with_split = assign_split(
        raw_events,
        LABEL_COLUMN,
        validation_size=VALIDATION_SIZE,
        test_size=TEST_SIZE,
    )
    display(events_with_split.groupby(["split", LABEL_COLUMN]).size().rename("events"))
else:
    print("Split assignment waits for dataset inspection and label confirmation.")

## Next: cleaning and physics-led EDA

Once the private data arrives, the next contribution will document the real label, event identifier, optional MC weight, and Belle II feature mapping. Cleaning and EDA sections will then be implemented here and in `src/` together before any model is trained.